# EDA Brand Logo

Exploratory analysis for logo detection datasets.

Steps:
- Inventory LogoDet-3K categories and brand folders.
- Sample image files to verify layout.
- Run the training data audit and summarize outputs.



In [ ]:
from __future__ import annotations

import json
import os
import sys
import subprocess
from pathlib import Path

# Resolve repo root from the notebook location.
REPO_ROOT = Path.cwd()
for parent in [REPO_ROOT] + list(REPO_ROOT.parents):
    if (parent / 'scripts').exists() and (parent / 'notebooks').exists():
        REPO_ROOT = parent
        break

# Ensure local modules are importable.
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'src'))

PY = sys.executable

def run(cmd: list[str]) -> None:
    # Run a command from the repo root with PYTHONPATH set.
    env = os.environ.copy()
    env['PYTHONPATH'] = os.pathsep.join([str(REPO_ROOT / 'src'), str(REPO_ROOT)])
    print('$', ' '.join(cmd))
    subprocess.run(cmd, cwd=str(REPO_ROOT), check=True, env=env)

def show_json(rel_path: str) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    try:
        data = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        print(path.read_text(encoding='utf-8', errors='ignore')[:2000])
        return
    print(json.dumps(data, indent=2))

def list_dir(rel_path: str, limit: int = 20) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    print(f'\n{rel_path}/')
    for item in sorted(path.iterdir())[:limit]:
        print(' -', item.name)


In [ ]:
from collections import Counter
from pathlib import Path

brand_root = REPO_ROOT / 'data' / 'raw' / 'brand' / 'LogoDet-3K'
summary = {
    'brand_root': str(brand_root),
    'category_counts': {},
    'sample_images': [],
}

print('Brand root:', brand_root)
if not brand_root.exists():
    print('Missing:', brand_root)
else:
    categories = [d for d in sorted(brand_root.iterdir()) if d.is_dir()]
    print('Top-level categories:', len(categories))
    for cat in categories:
        brands = [d for d in cat.iterdir() if d.is_dir()]
        summary['category_counts'][cat.name] = len(brands)
    for cat, count in list(summary['category_counts'].items())[:10]:
        print(f'{cat}: {count} brands')


In [ ]:
# Sample image files without scanning the entire dataset.
image_exts = {'.jpg', '.jpeg', '.png'}
if brand_root.exists():
    max_files = 20000
    samples = []
    seen = 0
    for path in brand_root.rglob('*'):
        if not path.is_file():
            continue
        if path.suffix.lower() not in image_exts:
            continue
        if seen < max_files:
            seen += 1
        else:
            break
        if len(samples) < 12:
            samples.append(str(path.relative_to(REPO_ROOT)))
    summary['sample_images'] = samples
    print('Sample images:')
    for item in samples:
        print(' -', item)


In [ ]:
# Persist summary for quick reference.
report_dir = REPO_ROOT / 'reports'
report_dir.mkdir(parents=True, exist_ok=True)
summary_path = report_dir / 'eda_brand_logo_summary.json'
summary_path.write_text(json.dumps(summary, indent=2))
print('Saved summary to', summary_path)


In [ ]:
# Generate a training data audit
run([PY, 'scripts/training_data_audit.py'])



In [ ]:
# Summarize brand-related entries from the training data audit.
audit_path = REPO_ROOT / 'reports' / 'TRAINING_DATA.json'
if not audit_path.exists():
    print('Missing:', audit_path)
else:
    audit = json.loads(audit_path.read_text(encoding='utf-8'))
    items = [
        item for item in audit.get('required', []) + audit.get('optional', [])
        if 'brand' in str(item.get('name', '')).lower()
    ]
    if not items:
        print('No brand entries found in TRAINING_DATA.json')
    else:
        print('brand datasets in audit:')
        for item in items:
            print(' -', item.get('name'), '|', item.get('status'), '|', item.get('path'))


In [ ]:
# Quick artifact index for verification.
for folder in ['models', 'experiments', 'artifacts', 'runs', 'reports', 'logs']:
    path = REPO_ROOT / folder
    if not path.exists():
        continue
    print(f'\n{folder}/')
    for item in sorted(path.iterdir())[:20]:
        print(' -', item.name)
